# Sovereign Cross-Currency Swap Valuation Engine
## How the Australian Government Hedges a USD Bond Issuance

---

### Context

Sovereign governments routinely issue bonds in foreign currencies to diversify their
investor base and tap deeper capital markets. Australia, for example, regularly issues
USD-denominated bonds via the Australian Office of Financial Management (AOFM).

But Australia's liabilities are in AUD — so it cannot simply hold USD debt. The
solution is a **cross-currency swap (CCS)**:

| Leg | Australia receives | Australia pays |
|-----|--------------------|----------------|
| USD | USD fixed coupon   | — |
| AUD | —                  | AUD floating (BBSW + basis) |

**Principal is exchanged** at inception (at spot rate) and reversed at maturity,
effectively converting the USD bond into synthetic AUD floating-rate debt.

### Trade Scenario

| Parameter | Value |
|-----------|-------|
| USD notional | USD 5,000,000,000 |
| AUD notional | AUD 7,750,000,000 |
| USD fixed coupon | 4.00% (semi-annual) |
| AUD floating | BBSW + (−25bp basis) |
| FX spot | 1.5500 AUD/USD |
| Tenor | 5 years |

### Key Maths

$$\text{MTM}_{AUD} = \text{PV}_{USD} \times S - \text{PV}_{AUD}$$

$$\text{PV}_{USD} = \sum_i CF^{USD}(t_i) \cdot D_{USD}(t_i)$$

$$\text{PV}_{AUD} = \sum_i CF^{AUD}(t_i) \cdot D_{AUD}(t_i)$$

$$F(t) = S \cdot \frac{D_{AUD}(t)}{D_{USD}(t)} \quad \text{(covered interest rate parity)}$$

## 1 · Imports and Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import market_data as md
from xccy_engine import (
    build_usd_curve, build_aud_curve,
    generate_schedule, usd_cashflows, aud_cashflows,
    fx_forward,
    pv_leg, mtm_ccs,
    shift_curve, dv01_usd, dv01_aud, fx_delta, scenario_grid,
)

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

print(f'USD market data: {md._usd_source}')
print(f'AUD market data: {md._aud_source}')

## 2 · Market Data

In [ ]:
print('USD Deposits:')
for t, r in md.usd_deposits:
    print(f'  {t:.2f}Y  {r:.3%}')

print('\nUSD Swaps:')
for t, r in md.usd_swaps:
    print(f'  {t:.1f}Y  {r:.3%}')

print('\nAUD Deposits:')
for t, r in md.aud_deposits:
    print(f'  {t:.2f}Y  {r:.3%}')

print('\nAUD Swaps:')
for t, r in md.aud_swaps:
    print(f'  {t:.1f}Y  {r:.3%}')

print(f'\nFX spot:  1 USD = {md.fx_spot:.4f} AUD')
print(f'CCS basis: {int(md.basis * 10000):+d}bp')

## 3 · Build Discount Curves

Both curves are bootstrapped from deposits (short end) and par swap rates (long end)
using **sequential bootstrapping** — the industry standard.

The DiscountCurve class uses **log-linear interpolation** between pillars:

$$\ln D(t) = \ln D(t_i) + \frac{t - t_i}{t_{i+1} - t_i} (\ln D(t_{i+1}) - \ln D(t_i))$$

This guarantees positive discount factors and piecewise-constant instantaneous forward rates.

In [ ]:
usd_curve = build_usd_curve()
aud_curve = build_aud_curve()

times = np.linspace(0.25, 5.0, 200)

usd_data = usd_curve.sample(times)
aud_data = aud_curve.sample(times)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Bootstrapped Discount Curves — USD (SOFR) and AUD (BBSW)', fontweight='bold')

# Zero rates
axes[0].plot(usd_data['times'], usd_data['zero_rates'] * 100,
             color='steelblue', linewidth=2, label='USD (SOFR)')
axes[0].plot(aud_data['times'], aud_data['zero_rates'] * 100,
             color='darkorange', linewidth=2, label='AUD (BBSW)')
# Market pillars
for t, r in md.usd_deposits + md.usd_swaps:
    axes[0].scatter(t, usd_curve.zero_rate(t) * 100, color='steelblue', s=40, zorder=5)
for t, r in md.aud_deposits + md.aud_swaps:
    axes[0].scatter(t, aud_curve.zero_rate(t) * 100, color='darkorange', s=40, zorder=5)
axes[0].set_title('Zero Rates')
axes[0].set_xlabel('Maturity (years)')
axes[0].set_ylabel('Zero rate (%)')
axes[0].legend()

# Discount factors
axes[1].plot(usd_data['times'], usd_data['discounts'],
             color='steelblue', linewidth=2, label='USD')
axes[1].plot(aud_data['times'], aud_data['discounts'],
             color='darkorange', linewidth=2, label='AUD')
axes[1].set_title('Discount Factors')
axes[1].set_xlabel('Maturity (years)')
axes[1].set_ylabel('D(t)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4 · Cashflow Schedule

Both legs exchange **principal at inception and maturity** (at the spot rate), plus
periodic payments:

- **USD leg** (received): fixed coupon = `N_USD × coupon × dt`
- **AUD leg** (paid): floating = `N_AUD × (BBSW_fwd + basis) × dt`

The BBSW forward rate for period $[t_{i-1}, t_i]$ is extracted from the AUD discount curve:

$$\text{BBSW}_{\text{fwd}} = \frac{e^{f(t_{i-1}, t_i) \cdot \Delta t} - 1}{\Delta t}$$

In [ ]:
schedule = generate_schedule(md.tenor, md.freq)
usd_cfs  = usd_cashflows(schedule, md.N_USD, md.coupon)
aud_cfs  = aud_cashflows(schedule, md.N_AUD, aud_curve, md.basis)

# Compute BBSW forwards for display
bbsw_fwds = []
prev = 0.0
for t in schedule:
    dt = t - prev
    f  = aud_curve.forward_rate(prev, t)
    bbsw_fwds.append((np.exp(f * dt) - 1.0) / dt)
    prev = t

rows = []
for i, t in enumerate(schedule):
    rows.append({
        'Date (Y)': t,
        'USD Fixed CF': usd_cfs[i][1],
        'BBSW Fwd (%)': bbsw_fwds[i] * 100,
        'AUD Float CF': aud_cfs[i][1],
    })

df_cf = pd.DataFrame(rows)

styled = df_cf.style\
    .format({
        'Date (Y)':    '{:.2f}',
        'USD Fixed CF': '{:,.0f}',
        'BBSW Fwd (%)': '{:.4f}%',
        'AUD Float CF': '{:,.0f}',
    })\
    .set_caption('Semi-annual cashflows — USD received, AUD paid')\
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-weight', 'bold'), ('font-size', '13px')]}])

styled

## 5 · Valuation: Price Each Leg and Compute MTM

$$\text{MTM}_{AUD} = \underbrace{\text{PV}_{USD} \times S}_{\text{USD leg in AUD}} - \underbrace{\text{PV}_{AUD}}_{\text{AUD leg}}$$

At **inception** with at-market rates, MTM ≈ 0 (zero-cost hedge). The small residual
here reflects the cross-currency basis cost — Australia pays BBSW − 25bp, accepting a
lower AUD floating rate in exchange for a lower headline USD coupon.

In [ ]:
pv_usd   = pv_leg(usd_cfs, usd_curve)
pv_aud   = pv_leg(aud_cfs, aud_curve)
mtm_base = pv_usd * md.fx_spot - pv_aud

print(f'PV (USD leg)         USD  {pv_usd:>20,.0f}')
print(f'PV (AUD leg)         AUD  {pv_aud:>20,.0f}')
print(f'PV_USD × FX_spot     AUD  {pv_usd * md.fx_spot:>20,.0f}')
print(f'─' * 50)
print(f'MTM (AUD)            AUD  {mtm_base:>+20,.0f}')
print()
print(f'MTM as % of AUD notional: {mtm_base / md.N_AUD * 100:.4f}%')
print()

# FX forward curve
fwd_times = np.array(schedule)
fwd_rates = [fx_forward(t,
                        spot=md.fx_spot,
                        df_aud=aud_curve.discount(t),
                        df_usd=usd_curve.discount(t))
             for t in fwd_times]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fwd_times, fwd_rates, 'o-', color='teal', linewidth=2)
ax.axhline(md.fx_spot, color='gray', linestyle='--', alpha=0.6, label=f'Spot = {md.fx_spot}')
ax.set_title('FX Forward Curve  (Covered Interest Rate Parity)', fontweight='bold')
ax.set_xlabel('Maturity (years)')
ax.set_ylabel('AUD / USD')
ax.legend()
plt.tight_layout()
plt.show()

print('\nFX Forward rates by maturity:')
for t, f in zip(fwd_times, fwd_rates):
    print(f'  {t:.2f}Y  {f:.6f}  (spot {f - md.fx_spot:+.6f})')

## 6 · Rate Sensitivity — DV01

**DV01** (Dollar Value of a Basis Point) is the MTM change for a +1bp parallel shift
of one yield curve, all else equal.

| Sensitivity | Intuition |
|-------------|----------|
| DV01 USD (negative) | Receiving fixed — when USD rates rise, the fixed USD leg we receive loses value |
| DV01 AUD (small, negative) | Paying BBSW − 25bp — almost duration-neutral (FRN), with a tiny spread-duration effect |

In [ ]:
common = dict(N_USD=md.N_USD, coupon=md.coupon,
              tenor=md.tenor, freq=md.freq, basis=md.basis)

dv01_u = dv01_usd(usd_curve, aud_curve, md.fx_spot, **common)
dv01_a = dv01_aud(usd_curve, aud_curve, md.fx_spot, **common)
fxd    = fx_delta(usd_curve, aud_curve, md.fx_spot, **common)

print(f'DV01 USD  AUD {dv01_u:>+15,.0f}  per +1bp USD rates')
print(f'DV01 AUD  AUD {dv01_a:>+15,.0f}  per +1bp AUD rates')
print(f'FX Delta  AUD {fxd:>+15,.0f}  per +1% USD appreciation')
print()
print(f'Total DV01 (USD equivalent, at spot):  USD {(dv01_u + dv01_a) / md.fx_spot:>+15,.0f}')

# Plot DV01 profile across shift sizes
shifts   = np.arange(-200, 205, 5)
pnl_usd  = np.array([mtm_ccs(shift_curve(usd_curve, s), aud_curve,  md.fx_spot, **common)
                     - mtm_base for s in shifts])
pnl_aud  = np.array([mtm_ccs(usd_curve, shift_curve(aud_curve, s),  md.fx_spot, **common)
                     - mtm_base for s in shifts])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('P&L vs Parallel Yield Curve Shift', fontweight='bold')

axes[0].plot(shifts, pnl_usd / 1e6, color='steelblue', linewidth=2)
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('USD Curve Shift')
axes[0].set_xlabel('Parallel shift (bp)')
axes[0].set_ylabel('P&L (AUD million)')

axes[1].plot(shifts, pnl_aud / 1e6, color='darkorange', linewidth=2)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title('AUD Curve Shift')
axes[1].set_xlabel('Parallel shift (bp)')
axes[1].set_ylabel('P&L (AUD million)')

plt.tight_layout()
plt.show()

## 7 · FX Sensitivity — MTM vs AUDUSD Spot

The FX exposure comes from converting the USD leg's PV back to AUD:

$$\frac{\partial \text{MTM}_{AUD}}{\partial S} = \text{PV}_{USD}$$

Australia **benefits** if the USD appreciates (more AUD per USD received).
Sovereigns typically hedge most FX exposure, retaining only residual basis risk.

In [ ]:
fx_range = np.linspace(1.10, 2.00, 200)
mtm_vs_fx = np.array([
    mtm_ccs(usd_curve, aud_curve, fx, **common) / 1e6
    for fx in fx_range
])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(fx_range, mtm_vs_fx, color='teal', linewidth=2)
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(md.fx_spot, color='gray', linestyle='--', alpha=0.6,
           label=f'Current spot: {md.fx_spot}')
ax.scatter([md.fx_spot], [mtm_base / 1e6], color='red', s=80, zorder=5,
           label=f'Base MTM: AUD {mtm_base/1e6:.0f}M')
ax.set_title('MTM vs AUDUSD Spot Rate', fontweight='bold')
ax.set_xlabel('AUD / USD')
ax.set_ylabel('MTM (AUD million)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'FX Delta (per 1% USD appreciation):  AUD {fxd:>+15,.0f}')
print(f'Analytical FX Delta = PV_USD:         USD {pv_usd:>15,.0f}')
print(f'  → in AUD: USD {pv_usd:,.0f} × {md.fx_spot:.4f} × 1% = AUD {pv_usd * md.fx_spot * 0.01:>+15,.0f}')

## 8 · Scenario Grid — Rates × FX

A 3 × 3 rate scenario grid (USD shift × AUD shift) repeated across three FX moves.
All values show **ΔMtM in AUD millions** relative to the base case.

**Key observation**: The CCS is much more sensitive to USD rates (DV01 ≈ −AUD 3.5M/bp)
than to AUD rates (DV01 ≈ −AUD 23K/bp), because the USD fixed leg has duration
while the AUD floating leg is nearly duration-neutral.

In [ ]:
grids = scenario_grid(
    usd_curve=usd_curve, aud_curve=aud_curve, fx_spot=md.fx_spot,
    usd_shifts=[-100, 0, 100],
    aud_shifts=[-100, 0, 100],
    fx_moves=[-0.10, 0.0, 0.10],
    **common,
)

for fx_label, df in grids.items():
    print(f'\n{fx_label}')
    print(df.round(1).to_string())

# Visualise as heatmaps
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('ΔMtM (AUD millions) — Rate Scenario × FX Move', fontweight='bold')

import matplotlib.colors as mcolors

for ax, (fx_label, df) in zip(axes, grids.items()):
    vals = df.values.astype(float)
    vmax = max(abs(vals.min()), abs(vals.max()))
    im = ax.imshow(vals, cmap='RdYlGn', aspect='auto',
                   vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=8)
    ax.set_title(fx_label)
    for (r, c), v in np.ndenumerate(vals):
        ax.text(c, r, f'{v:.0f}', ha='center', va='center', fontsize=9,
                color='black')
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.show()

## 9 · Summary and Key Takeaways

---

### What we built

A **self-contained Python valuation engine** for a sovereign cross-currency swap:

| Module | Responsibility |
|--------|---------------|
| `discount_curve.py` | Bootstrap from deposits + swaps, log-linear interpolation |
| `curves.py` | Build USD (SOFR) and AUD (BBSW) curves from live FRED data |
| `cashflows.py` | USD fixed leg, AUD floating BBSW + basis |
| `valuation.py` | PV each leg, MTM = PV_USD × FX − PV_AUD |
| `risk.py` | DV01 USD, DV01 AUD, FX delta, scenario grid |
| `market_data.py` | Live FRED fetch with hardcoded 2025 fallback |

### Key risk insights

| Risk factor | DV01 / Delta | Intuition |
|------------|-------------|----------|
| USD rates (+1bp) | ≈ −AUD 3.5M | Receiving fixed: duration ≈ 4.5Y on USD 5bn notional |
| AUD rates (+1bp) | ≈ −AUD 23K | Paying floating: near zero duration (FRN), only spread-duration |
| FX spot (+1%) | ≈ +AUD 670K | USD leg PV converted at higher spot = more AUD |

### Cross-currency basis

The **−25bp AUD/USD basis** reflects the premium offshore investors demand to lend USD
and receive AUD. For Australia, paying BBSW − 25bp means its synthetic AUD funding is
cheaper than a direct AUD bond issuance — a real funding benefit.

### FX forward structure

By CIP, FX forwards are fully determined by the interest rate differential. If AUD
rates are lower than USD rates, the AUD trades at a **forward premium** — confirming
the market expects AUD appreciation relative to USD over 5 years.

---

*Built with Python · No Bloomberg required · Data from FRED (St. Louis Fed)*